In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [2]:
train_df = pd.read_csv("data/train_v2.csv")
train_df.head()

,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1


In [3]:
train_df.shape

(970960, 2)

In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 970960 entries, 0 to 970959
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   msno      970960 non-null  str  
 1   is_churn  970960 non-null  int64
dtypes: int64(1), str(1)
memory usage: 55.6 MB


In [5]:
logs_df = pd.read_csv("data/user_logs_v2.csv", nrows=5)
logs_df.columns

Index(['msno', 'date', 'num_25', 'num_50', 'num_75', 'num_985', 'num_100',
       'num_unq', 'total_secs'],
      dtype='str')

In [6]:
sample_df, _ = train_test_split(
    train_df,
    train_size=50000,
    stratify=train_df['is_churn'],
    random_state=42
)

print("Original Ratio")
print(train_df['is_churn'].value_counts(normalize=True))

print("\nSample Ratio")
print(sample_df['is_churn'].value_counts(normalize=True))

Original Ratio
is_churn
0    0.910058
1    0.089942
Name: proportion, dtype: float64

Sample Ratio
is_churn
0    0.91006
1    0.08994
Name: proportion, dtype: float64


In [10]:
import os

# Setup
input_path = 'data/user_logs_v2.csv'
output_dir = 'data/processed/'
output_path = os.path.join(output_dir, 'sampled_logs.csv')

os.makedirs(output_dir, exist_ok=True)

target_set = set(sample_df['msno'])
print(f"Target cohort size: {len(target_set):,} unique users")

chunk_size = 500_000
toral_rows = 18_396_362
total_chunks = (toral_rows // chunk_size) +1

if os.path.exists(output_path):
    os.remove(output_path)

first_write = True
total_matched_rows = 0

# Ingestion & filtering loop
with tqdm(total=total_chunks, desc="Streaming and Filtering logs") as pbar:
    for chunk in pd.read_csv(input_path, chunksize=chunk_size):
        filtered_chunk = chunk[chunk['msno'].isin(target_set)]

        if not filtered_chunk.empty:
            total_matched_rows += len(filtered_chunk)
            filtered_chunk.to_csv(
                output_path,
                mode="a",
                header=first_write,
                index=False
            )
            first_write = False

        pbar.update(1)

print(f"\n Pipeline Completed Successfully!")
print(f"Total Rows Extracted: {total_matched_rows:,}")
print(f"Output File Size: {os.path.getsize(output_path) / (1024 * 1024): .2f} MB")

Target cohort size: 50,000 unique users


Streaming and Filtering logs: 100%|██████████| 37/37 [01:40<00:00,  2.73s/it]


 Pipeline Completed Successfully!
Total Rows Extracted: 695,320
Output File Size:  51.45 MB


In [11]:
sample_logs = pd.read_csv("data/processed/sampled_logs.csv")
sample_logs['msno'].nunique()

38750